In [2]:
import numpy as np
import torch
import torch.nn.functional as F

def load_vectors(file_path, max_words=None):
    """
    Načte textový soubor s word embeddings.
    Pokud je max_words nastaveno, načte pouze prvních max_words slov.
    Vrací: seznam slov, slovník word2idx a L2 normalizovanou matici vektorů.
    """
    words = []
    word2idx = {}
    vectors = []
    
    print(f"Načítám {file_path}...")
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if max_words is not None and i >= max_words:
                break
                
            parts = line.strip().split()
            if not parts: continue
                
            word = parts[0]
            vec = np.array(parts[1:], dtype=np.float32)
            
            words.append(word)
            word2idx[word] = i
            vectors.append(vec)
            
    vecs = np.array(vectors)
    vecs_norm = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)
    
    print(f"Načteno {len(words)} slov, dimenze {vecs_norm.shape[1]}")
    return words, word2idx, vecs_norm

In [ ]:
# Načtení celého GloVe
glove_words, glove_word2idx, glove_vecs = load_vectors("glove.6B.50d.txt")

def cossim(a, b):
    """Vrátí kosinovou podobnost dvou 1D numpy polí."""
    if isinstance(a, torch.Tensor): a = a.numpy()
    if isinstance(b, torch.Tensor): b = b.numpy()
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def get_analogy(word_a, word_b, word_c, words, word2idx, vecs_norm, top_k=5):
    """word_a - word_b + word_c = ?"""
    for w in [word_a, word_b, word_c]:
        if w not in word2idx:
            print(f"Slovo '{w}' není ve slovníku.")
            return []
            
    idx_a, idx_b, idx_c = word2idx[word_a], word2idx[word_b], word2idx[word_c]
    
    # Výpočet vektoru
    target_vec = vecs_norm[idx_a] - vecs_norm[idx_b] + vecs_norm[idx_c]
    
    # Skalární součin s normalizovanou maticí = kosinová podobnost ke všem slovům
    sims = vecs_norm @ target_vec
    
    # Odstraníme vstupní slova z výsledků
    sims[[idx_a, idx_b, idx_c]] = -np.inf
    
    # Získáme top K indexů
    best_indices = np.argsort(sims)[::-1][:top_k]
    return [(words[i], sims[i]) for i in best_indices]

print("\n--- Test analogií v GloVe ---")
print("king - man + woman = ?")
results = get_analogy("king", "man", "woman", glove_words, glove_word2idx, glove_vecs)
for w, sim in results:
    print(f"  {w} ({sim:.4f})")
print("purple - blue + yellow = ?")
results = get_analogy("purple", "blue", "yellow", glove_words, glove_word2idx, glove_vecs)
for w, sim in results:
    print(f"  {w} ({sim:.4f})")
print("orange - yellow + blue = ?")
results = get_analogy("orange", "yellow", "blue", glove_words, glove_word2idx, glove_vecs)
for w, sim in results:
    print(f"  {w} ({sim:.4f})")

Načítám glove.6B.50d.txt...
Načteno 400001 slov, dimenze 50

--- Test analogií v GloVe ---
king - man + woman = ?
  queen (0.8475)
  throne (0.7621)
  prince (0.7549)
  daughter (0.7432)
  elizabeth (0.7418)
purple - blue + yellow = ?
  pink (0.9151)
  ribbons (0.8913)
  flowers (0.8900)
  reddish (0.8900)
  leaf (0.8819)
king - man + woman = ?
  queen (0.8475)
  throne (0.7621)
  prince (0.7549)
  daughter (0.7432)
  elizabeth (0.7418)


In [ ]:
# 30 000 slov 
models = {
    "GloVe": load_vectors("glove.6B.50d.txt", max_words=30000),
    "Skip-gram": load_vectors("vectors_skipgram.txt", max_words=30000),
    "CBOW": load_vectors("vectors_cbow.txt", max_words=30000)
}

def evaluate_analogies(questions_file, word2idx, vecs_norm):
    semantic_correct, semantic_total = 0, 0
    syntactic_correct, syntactic_total = 0, 0
    is_semantic = True
    
    try:
        with open(questions_file, 'r', encoding='utf-8') as f:
            for line in f:
                if line.startswith(':'):
                    is_semantic = 'gram' not in line.lower()
                    continue
                    
                parts = line.strip().lower().split()
                if len(parts) != 4: continue
                a, b, c, expected = parts
                
                if not all(w in word2idx for w in (a, b, c, expected)):
                    continue
                    
                idx_a, idx_b, idx_c, idx_exp = word2idx[a], word2idx[b], word2idx[c], word2idx[expected]
                
                # target = b - a + c 
                target_vec = vecs_norm[idx_b] - vecs_norm[idx_a] + vecs_norm[idx_c]
                
                sims = vecs_norm @ target_vec
                sims[[idx_a, idx_b, idx_c]] = -np.inf
                
                predicted_idx = np.argmax(sims)
                is_correct = (predicted_idx == idx_exp)
                
                if is_semantic:
                    semantic_total += 1
                    semantic_correct += int(is_correct)
                else:
                    syntactic_total += 1
                    syntactic_correct += int(is_correct)
    except FileNotFoundError:
        print(f"Soubor {questions_file} nenalezen!")
        return 0, 0
                
    sem_acc = semantic_correct / semantic_total if semantic_total > 0 else 0
    syn_acc = syntactic_correct / syntactic_total if syntactic_total > 0 else 0
    return sem_acc, syn_acc

Načítám glove.6B.50d.txt...
Načteno 30000 slov, dimenze 50
Načítám vectors_skipgram.txt...
Načteno 30000 slov, dimenze 300
Načítám vectors_cbow.txt...
Načteno 30000 slov, dimenze 300


In [8]:
print("--- Porovnání přesnosti na questions-words.txt ---")
for name, (w, w2i, vecs) in models.items():
    sem_acc, syn_acc = evaluate_analogies("questions-words.txt", w2i, vecs)
    print(f"{name:10} -> Sémantická přesnost: {sem_acc:7.2%} | Syntaktická přesnost: {syn_acc:7.2%}")

--- Porovnání přesnosti na questions-words.txt ---
GloVe      -> Sémantická přesnost:  50.87% | Syntaktická přesnost:  48.90%
Skip-gram  -> Sémantická přesnost:  50.26% | Syntaktická přesnost:  31.79%
CBOW       -> Sémantická přesnost:   9.48% | Syntaktická přesnost:  19.55%


In [6]:
def get_top_k_closest(word, words, word2idx, vecs_norm, top_k=5):
    if word not in word2idx: return []
    idx = word2idx[word]
    vec = vecs_norm[idx]
    
    sims = vecs_norm @ vec
    sims[idx] = -np.inf # Vyřadíme samotné hledané slovo
    
    best_indices = np.argsort(sims)[::-1][:top_k]
    return [words[i] for i in best_indices]

target_words = ["king", "france", "computer", "quickly", "violin"]

print("--- 5 nejpodobnějších slov ---")
for word in target_words:
    print(f"\nSlovo: {word.upper()}")
    for name, (w_list, w2i, vecs) in models.items():
        closest = get_top_k_closest(word, w_list, w2i, vecs)
        print(f"  {name:10}: {', '.join(closest)}")

--- 5 nejpodobnějších slov ---

Slovo: KING
  GloVe     : prince, queen, ii, emperor, son
  Skip-gram : kings, throne, prince, queen, crowned
  CBOW      : prince, pope, emperor, throne, anjou

Slovo: FRANCE
  GloVe     : french, belgium, paris, spain, netherlands
  Skip-gram : french, paris, spain, italy, germany
  CBOW      : spain, french, hungary, germany, europe

Slovo: COMPUTER
  GloVe     : computers, software, technology, electronic, internet
  Skip-gram : computers, hardware, computing, software, graphics
  CBOW      : computers, computing, technology, software, hardware

Slovo: QUICKLY
  GloVe     : soon, again, eventually, turn, finally
  Skip-gram : soon, eventually, rapidly, easily, effectively
  CBOW      : soon, eventually, slowly, gradually, rapidly

Slovo: VIOLIN
  GloVe     : cello, piano, clarinet, concerto, flute
  Skip-gram : cello, piano, concerto, viola, sonata
  CBOW      : clarinets, banjo, plectrum, dulcimer, clarinet


In [ ]:
"""
--- Porovnání přesnosti na questions-words.txt ---
GloVe      -> Sémantická přesnost:  50.87% | Syntaktická přesnost:  48.90%
Skip-gram  -> Sémantická přesnost:  50.26% | Syntaktická přesnost:  31.79%
CBOW       -> Sémantická přesnost:   9.48% | Syntaktická přesnost:  19.55%

--- 5 nejpodobnějších slov ---

Slovo: KING
  GloVe     : prince, queen, ii, emperor, son
  Skip-gram : kings, throne, prince, queen, crowned
  CBOW      : prince, pope, emperor, throne, anjou

Slovo: FRANCE
  GloVe     : french, belgium, paris, spain, netherlands
  Skip-gram : french, paris, spain, italy, germany
  CBOW      : spain, french, hungary, germany, europe

Slovo: COMPUTER
  GloVe     : computers, software, technology, electronic, internet
  Skip-gram : computers, hardware, computing, software, graphics
  CBOW      : computers, computing, technology, software, hardware

Slovo: QUICKLY
  GloVe     : soon, again, eventually, turn, finally
  Skip-gram : soon, eventually, rapidly, easily, effectively
  CBOW      : soon, eventually, slowly, gradually, rapidly

Slovo: VIOLIN
  GloVe     : cello, piano, clarinet, concerto, flute
  Skip-gram : cello, piano, concerto, viola, sonata
  CBOW      : clarinets, banjo, plectrum, dulcimer, clarinet


GloVe:
Nejlepší, největší dataset, zachycuje kontext nejlépe.

Skip-gram:
V sémantice je skoro stejně dobrý jako GloVe, protože se z jednoho slova snaží trefit celý kontext, takže má mnohem víc updatů pro zachycení významu.

CBOW:
Malý dataset, průměruje celé okolí slova do jednoho vektoru, čímž ztrácí detaily a nedokáže pořádně naučit kontext.
"""